In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("olist.db")

print("SQLite database connected successfully!")

SQLite database connected successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving archive (1).zip to archive (1).zip


In [ ]:
import zipfile
import os

zip_file = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall("olist_data")

print("Dataset extracted successfully!")
print(os.listdir("olist_data"))

Dataset extracted successfully!
['olist_order_reviews_dataset.csv', 'olist_sellers_dataset.csv', 'olist_products_dataset.csv', 'olist_order_items_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_payments_dataset.csv', 'product_category_name_translation.csv', 'olist_customers_dataset.csv', 'olist_orders_dataset.csv']


In [ ]:
import pandas as pd
import os

for file in os.listdir("olist_data"):
    if file.endswith(".csv"):
        df = pd.read_csv(f"olist_data/{file}")
        table_name = file.replace(".csv", "").replace("olist_", "").replace("_dataset", "")
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"{table_name} imported successfully")

order_reviews imported successfully
sellers imported successfully
products imported successfully
order_items imported successfully
geolocation imported successfully
order_payments imported successfully
product_category_name_translation imported successfully
customers imported successfully
orders imported successfully


In [ ]:
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")

tables_list = cursor.fetchall()

for table in tables_list:
    print(table[0])

order_reviews
sellers
products
order_items
geolocation
order_payments
product_category_name_translation
customers
orders


In [ ]:
query = """
SELECT COUNT(*) AS total_orders
FROM orders;
"""

pd.read_sql_query(query, conn)

,total_orders
0,99441


In [ ]:
query = """
SELECT COUNT(*) AS delivered_orders
FROM orders
WHERE order_status = 'delivered';
"""

pd.read_sql_query(query, conn)

,delivered_orders
0,96478


In [ ]:
query = """
SELECT order_status, COUNT(*) AS order_count
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;
"""

pd.read_sql_query(query, conn)

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [ ]:
query = """
SELECT
    strftime('%Y-%m', order_purchase_timestamp) AS month,
    COUNT(*) AS total_orders
FROM orders
GROUP BY month
ORDER BY month;
"""

pd.read_sql_query(query, conn)

,month,total_orders
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


In [ ]:
query = """
SELECT
    t.product_category_name_english AS category,
    ROUND(SUM(i.price), 2) AS revenue
FROM order_items i
JOIN products p
    ON i.product_id = p.product_id
JOIN product_category_name_translation t
    ON p.product_category_name = t.product_category_name
GROUP BY category
ORDER BY revenue DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,category,revenue
0,health_beauty,1258681.34
1,watches_gifts,1205005.68
2,bed_bath_table,1036988.68
3,sports_leisure,988048.97
4,computers_accessories,911954.32
5,furniture_decor,729762.49
6,cool_stuff,635290.85
7,housewares,632248.66
8,auto,592720.11
9,garden_tools,485256.46


In [ ]:
query = """
SELECT
    t.product_category_name_english AS category,
    COUNT(DISTINCT i.order_id) AS total_orders
FROM order_items i
JOIN products p
    ON i.product_id = p.product_id
JOIN product_category_name_translation t
    ON p.product_category_name = t.product_category_name
GROUP BY category
ORDER BY total_orders DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,category,total_orders
0,bed_bath_table,9417
1,health_beauty,8836
2,sports_leisure,7720
3,computers_accessories,6689
4,furniture_decor,6449
5,housewares,5884
6,watches_gifts,5624
7,telephony,4199
8,auto,3897
9,toys,3886


In [ ]:
query = """
SELECT
    t.product_category_name_english AS category,
    ROUND(SUM(i.price) / COUNT(DISTINCT i.order_id), 2) AS avg_order_value
FROM order_items i
JOIN products p
    ON i.product_id = p.product_id
JOIN product_category_name_translation t
    ON p.product_category_name = t.product_category_name
GROUP BY category
ORDER BY avg_order_value DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,category,avg_order_value
0,computers,1231.84
1,small_appliances_home_oven_and_coffee,632.61
2,home_appliances_2,484.26
3,agro_industry_and_commerce,398.52
4,musical_instruments,304.93
5,small_appliances,302.62
6,fixed_telephony,274.58
7,construction_tools_safety,242.78
8,air_conditioning,217.49
9,office_furniture,215.21


In [ ]:
query = """
SELECT
    payment_type,
    COUNT(*) AS payment_records,
    ROUND(SUM(payment_value), 2) AS total_payment
FROM order_payments
GROUP BY payment_type
ORDER BY total_payment DESC;
"""

pd.read_sql_query(query, conn)

,payment_type,payment_records,total_payment
0,credit_card,76795,12542084.19
1,boleto,19784,2869361.27
2,voucher,5775,379436.87
3,debit_card,1529,217989.79
4,not_defined,3,0.00


In [ ]:
query = """
SELECT
    seller_id,
    ROUND(SUM(price), 2) AS total_revenue
FROM order_items
GROUP BY seller_id
ORDER BY total_revenue DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,seller_id,total_revenue
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63
1,53243585a1d6dc2643021fd1853d8905,222776.05
2,4a3ca9315b744ce9f8e9374361493884,200472.92
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03
4,7c67e1448b00f6e969d365cea6b010ab,187923.89
5,7e93a43ef30c4f03f38b393420bc753a,176431.87
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55
9,955fee9216a65b617aa5c0531780ce60,135171.70


In [ ]:
query = """
SELECT
    COUNT(*) AS repeat_customers
FROM (
    SELECT
        c.customer_unique_id,
        COUNT(o.order_id) AS total_orders
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
    HAVING COUNT(o.order_id) > 1
);
"""

pd.read_sql_query(query, conn)

,repeat_customers
0,2801


In [ ]:
query = """
SELECT
    COUNT(*) AS total_customers,
    SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END) AS repeat_customers,
    ROUND(
        100.0 * SUM(CASE WHEN order_count > 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS repeat_customer_rate
FROM (
    SELECT
        c.customer_unique_id,
        COUNT(o.order_id) AS order_count
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
);
"""

pd.read_sql_query(query, conn)

,total_customers,repeat_customers,repeat_customer_rate
0,93358,2801,3.0


In [ ]:
query = """
SELECT
    CASE
        WHEN julianday(order_delivered_customer_date) >
             julianday(order_estimated_delivery_date)
        THEN 'Late'
        ELSE 'On Time or Early'
    END AS delivery_status,
    COUNT(*) AS total_orders,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL
GROUP BY delivery_status
ORDER BY total_orders DESC;
"""

pd.read_sql_query(query, conn)

,delivery_status,total_orders,percentage
0,On Time or Early,88644,91.89
1,Late,7826,8.11


In [ ]:
query = """
SELECT
    CASE
        WHEN julianday(o.order_delivered_customer_date) >
             julianday(o.order_estimated_delivery_date)
        THEN 'Late'
        ELSE 'On Time or Early'
    END AS delivery_status,
    ROUND(AVG(r.review_score), 2) AS average_review_score,
    COUNT(*) AS total_reviews
FROM orders o
JOIN order_reviews r
    ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY delivery_status;
"""

pd.read_sql_query(query, conn)

,delivery_status,average_review_score,total_reviews
0,Late,2.57,7700
1,On Time or Early,4.29,88653


In [ ]:
query = """
SELECT
    c.customer_state,
    ROUND(SUM(i.price), 2) AS revenue
FROM order_items i
JOIN orders o
    ON i.order_id = o.order_id
JOIN customers c
    ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY revenue DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,customer_state,revenue
0,SP,5202955.05
1,RJ,1824092.67
2,MG,1585308.03
3,RS,750304.02
4,PR,683083.76
5,SC,520553.34
6,BA,511349.99
7,DF,302603.94
8,GO,294591.95
9,ES,275037.31


In [ ]:
query = """
SELECT
    customer_state,
    COUNT(DISTINCT customer_unique_id) AS total_customers
FROM customers
GROUP BY customer_state
ORDER BY total_customers DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,customer_state,total_customers
0,SP,40302
1,RJ,12384
2,MG,11259
3,RS,5277
4,PR,4882
5,SC,3534
6,BA,3277
7,DF,2075
8,ES,1964
9,GO,1952


In [ ]:
query = """
SELECT
    c.customer_state,
    COUNT(DISTINCT c.customer_unique_id) AS customers,
    ROUND(SUM(i.price), 2) AS total_revenue,
    ROUND(
        SUM(i.price) / COUNT(DISTINCT c.customer_unique_id),
        2
    ) AS revenue_per_customer
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items i
    ON o.order_id = i.order_id
GROUP BY c.customer_state
ORDER BY revenue_per_customer DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,customer_state,customers,total_revenue,revenue_per_customer
0,PB,516,115268.08,223.39
1,AC,77,15982.95,207.57
2,AL,399,80314.81,201.29
3,AP,67,13474.30,201.11
4,RO,235,46140.64,196.34
5,PA,944,178947.81,189.56
6,TO,272,49621.74,182.43
7,PI,481,86914.08,180.69
8,MT,873,156453.53,179.21
9,RN,471,83034.98,176.30


In [ ]:
query = """
SELECT
    c.customer_state,
    COUNT(*) AS total_delivered_orders,
    SUM(
        CASE
            WHEN julianday(o.order_delivered_customer_date) >
                 julianday(o.order_estimated_delivery_date)
            THEN 1
            ELSE 0
        END
    ) AS late_orders,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN julianday(o.order_delivered_customer_date) >
                     julianday(o.order_estimated_delivery_date)
                THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS late_delivery_rate
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY late_delivery_rate DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,customer_state,total_delivered_orders,late_orders,late_delivery_rate
0,AL,397,95,23.93
1,MA,717,141,19.67
2,PI,476,76,15.97
3,CE,1279,196,15.32
4,SE,335,51,15.22
5,BA,3256,457,14.04
6,RJ,12350,1664,13.47
7,TO,274,35,12.77
8,PA,946,117,12.37
9,ES,1995,244,12.23


In [ ]:
query = """
SELECT
    c.customer_state,
    COUNT(*) AS total_orders,
    SUM(
        CASE
            WHEN julianday(o.order_delivered_customer_date) >
                 julianday(o.order_estimated_delivery_date)
            THEN 1
            ELSE 0
        END
    ) AS late_orders
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY late_orders DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,customer_state,total_orders,late_orders
0,SP,40494,2387
1,RJ,12350,1664
2,MG,11354,637
3,BA,3256,457
4,RS,5344,382
5,SC,3546,346
6,PR,4923,246
7,ES,1995,244
8,CE,1279,196
9,PE,1593,172


In [ ]:
query = """
SELECT
    t.product_category_name_english AS category,
    ROUND(AVG(r.review_score), 2) AS avg_review_score,
    COUNT(*) AS total_reviews
FROM order_reviews r
JOIN order_items i
    ON r.order_id = i.order_id
JOIN products p
    ON i.product_id = p.product_id
JOIN product_category_name_translation t
    ON p.product_category_name = t.product_category_name
GROUP BY category
HAVING COUNT(*) >= 100
ORDER BY avg_review_score ASC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,category,avg_review_score,total_reviews
0,office_furniture,3.49,1687
1,fashion_male_clothing,3.64,131
2,fixed_telephony,3.68,262
3,audio,3.83,361
4,home_confort,3.83,435
5,construction_tools_safety,3.84,193
6,bed_bath_table,3.90,11137
7,furniture_decor,3.90,8331
8,furniture_living_room,3.90,502
9,computers_accessories,3.93,7849


In [ ]:
query = """
SELECT
    review_score,
    COUNT(*) AS total_reviews,
    ROUND(
        100.0 * COUNT(*) / (SELECT COUNT(*) FROM order_reviews),
        2
    ) AS percentage
FROM order_reviews
GROUP BY review_score
ORDER BY review_score;
"""

pd.read_sql_query(query, conn)

,review_score,total_reviews,percentage
0,1,11424,11.51
1,2,3151,3.18
2,3,8179,8.24
3,4,19142,19.29
4,5,57328,57.78


In [ ]:
query = """
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS month,
    ROUND(AVG(r.review_score), 2) AS avg_review_score,
    COUNT(*) AS total_reviews
FROM orders o
JOIN order_reviews r
    ON o.order_id = r.order_id
WHERE o.order_purchase_timestamp IS NOT NULL
GROUP BY month
ORDER BY month;
"""

pd.read_sql_query(query, conn)

,month,avg_review_score,total_reviews
0,2016-09,1.00,4
1,2016-10,3.57,321
2,2016-12,5.00,1
3,2017-01,4.07,797
4,2017-02,4.02,1776
5,2017-03,4.07,2676
6,2017-04,4.04,2394
7,2017-05,4.14,3703
8,2017-06,4.15,3250
9,2017-07,4.17,4032


In [ ]:
query = """
SELECT
    ROUND(SUM(price), 2) AS total_product_value,
    ROUND(SUM(freight_value), 2) AS total_freight,
    ROUND(100.0 * SUM(freight_value) / SUM(price), 2) AS freight_percentage
FROM order_items;
"""

pd.read_sql_query(query, conn)

,total_product_value,total_freight,freight_percentage
0,13591643.7,2251909.54,16.57


In [ ]:
query = """
SELECT
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(payment_value), 2) AS total_sales,
    ROUND(SUM(payment_value) / COUNT(DISTINCT order_id), 2) AS average_order_value
FROM order_payments;
"""

pd.read_sql_query(query, conn)

,total_orders,total_sales,average_order_value
0,99440,16008872.12,160.99
